In [0]:
-- Paso 1: MERGE incremental para la tabla cuentas_historico
-- Carga histórica con id_cuenta y fecha_extraccion como claves
MERGE INTO IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.cuentas_historico') AS target
USING (
    SELECT
        CAST(account_id AS BIGINT) AS id_cuenta,
        account_name AS nombre_cuenta,
        email AS correo_electronico,
        CAST(created_at AS TIMESTAMP) AS fecha_creacion,
        CAST(updated_at AS TIMESTAMP) AS fecha_actualizacion,
        fecha_extraccion
    FROM IDENTIFIER(:catalogo_origen || '.datavision_' || :alumno || '.accounts')
    WHERE fecha_extraccion = :fecha_extraccion
) AS source
ON target.id_cuenta = source.id_cuenta 
   AND target.fecha_extraccion = source.fecha_extraccion
WHEN MATCHED THEN
    UPDATE SET
        target.nombre_cuenta = source.nombre_cuenta,
        target.correo_electronico = source.correo_electronico,
        target.fecha_creacion = source.fecha_creacion,
        target.fecha_actualizacion = source.fecha_actualizacion
WHEN NOT MATCHED THEN
    INSERT (
        id_cuenta,
        nombre_cuenta,
        correo_electronico,
        fecha_creacion,
        fecha_actualizacion,
        fecha_extraccion
    )
    VALUES (
        source.id_cuenta,
        source.nombre_cuenta,
        source.correo_electronico,
        source.fecha_creacion,
        source.fecha_actualizacion,
        source.fecha_extraccion
    );

In [0]:
-- Paso 2: OVERWRITE completo de la tabla cuentas
-- Solo se ejecuta si el MERGE anterior fue exitoso
INSERT OVERWRITE TABLE IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.cuentas')
SELECT
    CAST(account_id AS BIGINT) AS id_cuenta,
    account_name AS nombre_cuenta,
    email AS correo_electronico,
    CAST(created_at AS TIMESTAMP) AS fecha_creacion,
    CAST(updated_at AS TIMESTAMP) AS fecha_actualizacion
FROM IDENTIFIER(:catalogo_origen || '.datavision_' || :alumno || '.accounts')
WHERE fecha_extraccion = :fecha_extraccion;

## Verificación de resultados

In [0]:
-- Verificar la cantidad de registros en cuentas_historico
SELECT
    'cuentas_historico' as tabla,
    COUNT(*) as total_registros,
    COUNT(DISTINCT id_cuenta) as cuentas_unicas,
    COUNT(DISTINCT fecha_extraccion) as fechas_extraccion,
    MAX(fecha_actualizacion) as ultima_actualizacion
FROM IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.cuentas_historico')

UNION ALL

-- Verificar la cantidad de registros en cuentas
SELECT
    'cuentas' as tabla,
    COUNT(*) as total_registros,
    COUNT(DISTINCT id_cuenta) as cuentas_unicas,
    NULL as fechas_extraccion,
    MAX(fecha_actualizacion) as ultima_actualizacion
FROM IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.cuentas');

In [0]:
-- Mostrar una muestra de los datos de cuentas_historico
SELECT 'cuentas_historico' as origen, *
FROM IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.cuentas_historico')
WHERE fecha_extraccion = :fecha_extraccion
ORDER BY fecha_actualizacion DESC
LIMIT 5;

In [0]:
-- Mostrar una muestra de los datos de cuentas
SELECT 'cuentas' as origen, *
FROM IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.cuentas')
ORDER BY fecha_actualizacion DESC
LIMIT 5;